# معمل المخرجات المهيكلة عبر `Dataclasses` وتكامل `create_agent`

أهلاً بك في المعمل العملي للمحاضرة 65!

في المعمل ده هنجمع الطرق التلاتة للمخرجات المهيكلة مع بعض، لكن المرة دي جوه **الوكيل الموحد `create_agent`** عن طريق معامل `response_format`:
1. **`Pydantic`**: تحقق صارم للأنواع + وصول بالخصائص (`res.name`).
2. **`TypedDict`**: قاموس خفيف بدون تحقق وقت التشغيل (`res["name"]`).
3. **`@dataclass`**: كلاس قياسي مدمج في بايثون من غير أي مكتبات خارجية + وصول مريح بالخصائص (`res.name`).


In [1]:
# تجهيز الموديل وأدوات الثلاث طرق
from dataclasses import dataclass
from typing_extensions import TypedDict
from pydantic import BaseModel
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_groq import ChatGroq

load_dotenv()
model = ChatGroq(model="openai/gpt-oss-20b", temperature=0.0)
query = {"messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]}
print("جاهزين نقارن التلات طرق داخل الوكيل الموحد!")


جاهزين نقارن التلات طرق داخل الوكيل الموحد!


### 1. تجربة `Pydantic` و `TypedDict` داخل `create_agent(..., response_format=...)`

لاحظ إن الوكيل بيرجع النتيجة المهيكلة في مفتاح مخصوص اسمه `structured_response` بجانب `messages`!


In [2]:
class ContactPydantic(BaseModel):
    name: str
    email: str
    phone: str

class ContactTypedDict(TypedDict):
    name: str
    email: str
    phone: str

res_p = create_agent(model=model, response_format=ContactPydantic).invoke(query)["structured_response"]
res_t = create_agent(model=model, response_format=ContactTypedDict).invoke(query)["structured_response"]

print("نتيجة Pydantic:", type(res_p).__name__, "->", res_p)
print("نتيجة TypedDict:", type(res_t).__name__, "->", res_t)


نتيجة Pydantic: ContactPydantic -> name='John Doe' email='john@example.com' phone='(555) 123-4567'
نتيجة TypedDict: dict -> {'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}


### 2. تجربة `@dataclass` القياسية داخل `create_agent`

ميزة `dataclass` إنها مدمجة في بايثون من إصدار 3.7، بتديك كائن أنيق تقدر تقرأه بالنقطة (`contact.email`) بس خفيف ومن غير قيود Validation صارمة.


In [3]:
@dataclass
class ContactDataclass:
    name: str
    email: str
    phone: str

agent_dc = create_agent(model=model, response_format=ContactDataclass)
result = agent_dc.invoke(query)

contact = result["structured_response"]
print("نوع المخرج:", type(contact).__name__)
print("البيانات المستخرجة:", contact)
print(f"الاسم: {contact.name} | الإيميل: {contact.email} | الموبايل: {contact.phone}")


نوع المخرج: ContactDataclass
البيانات المستخرجة: ContactDataclass(name='John Doe', email='john@example.com', phone='(555) 123-4567')
الاسم: John Doe | الإيميل: john@example.com | الموبايل: (555) 123-4567


### 3. جرب بنفسك!

دلوقتي دورك للتجربة الحرة:
1. صمم `@dataclass` جديدة خاصة بيك (مثلاً `JobApplicant` لاستخراج الاسم والسنوات وسنوات الخبرة والمهارة الأساسية من سطر سيرة ذاتية).
2. مررها لـ `create_agent(model=model, response_format=JobApplicant)` وشوف النتيجة المنسقة في `structured_response`!


In [4]:
# جرب بنفسك هنا
@dataclass
class JobApplicant:
    full_name: str
    years_of_experience: int
    primary_skill: str

cv_snippet = "My name is Sara Ahmed, I have 4 years of experience building RAG pipelines with LangGraph."
hr_agent = create_agent(model=model, response_format=JobApplicant)

hr_res = hr_agent.invoke({"messages": [{"role": "user", "content": f"Extract candidate profile from: {cv_snippet}"}]})
candidate = hr_res["structured_response"]

print("بيانات المتقدم:", candidate)
print(f"المرشح {candidate.full_name} لديه {candidate.years_of_experience} سنوات خبرة في {candidate.primary_skill}.")


بيانات المتقدم: JobApplicant(full_name='Sara Ahmed', years_of_experience=4, primary_skill='RAG pipelines with LangGraph')
المرشح Sara Ahmed لديه 4 سنوات خبرة في RAG pipelines with LangGraph.
